In [ ]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
import threading
# import threading
mt5.initialize()

In [ ]:

global df


def price_action(symbol, lot, ask, bid, order_type):
    buy_profit = mt5.order_calc_profit(order_type, symbol, lot, ask, bid)
    return buy_profit

def ema(s, n):
    ema = []
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)

    return ema

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df


def niss_Fast(src):
    emaF1 = ema(src, 3)
    emaF2 = ema(src, 5)
    emaF3 = ema(src, 7)
    emaF4 = ema(src, 9)
    emaF5 = ema(src, 11)
    emaF6 = ema(src, 13)
    emaF7 = ema(src, 15)
    emaF8 = ema(src, 17)
    emaF9 = ema(src, 19)
    emaF10 = ema(src, 21)
    emaF11 = ema(src, 23)
    return list(map(lambda x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11:(x1+x2+x3+x4+x5+x6+x7+x8+x9+x10+x11)/11, \
             emaF1[len(emaF1)-len(emaF11):], \
             emaF2[len(emaF2)-len(emaF11):],  emaF3[len(emaF3)-len(emaF11):], emaF4[len(emaF4)-len(emaF11):], \
            emaF5[len(emaF5)-len(emaF11):], emaF6[len(emaF6)-len(emaF11):], emaF7[len(emaF7)-len(emaF11):], \
            emaF8[len(emaF8)-len(emaF11):], emaF9[len(emaF9)-len(emaF11):], emaF10[len(emaF10)-len(emaF11):], \
            emaF11))
    
def niss_Slow(src):
    emaS1 = ema(src, 25)
    emaS2 = ema(src, 28)
    emaS3 = ema(src, 31)
    emaS4 = ema(src, 34)
    emaS5 = ema(src, 37)
    emaS6 = ema(src, 40)
    emaS7 = ema(src, 43)
    emaS8 = ema(src, 46)
    emaS9 = ema(src, 49)
    emaS10 = ema(src, 52)
    emaS11 = ema(src, 55)
    emaS12 = ema(src, 58)
    emaS13 = ema(src, 61)
    emaS14 = ema(src, 64)
    emaS15 = ema(src, 67)
    emaS16 = ema(src, 70)
    return list(map(lambda x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, x13, x14, x15, x16: \
                    (x1+x2+x3+x4+x5+x6+x7+x8+x9+x10+x11+x12+x13+x14+x15+x16)/16, \
         emaS1[len(emaS1)-len(emaS16):], \
         emaS2[len(emaS2)-len(emaS16):],  emaS3[len(emaS3)-len(emaS16):], emaS4[len(emaS4)-len(emaS16):], \
        emaS5[len(emaS5)-len(emaS16):], emaS6[len(emaS6)-len(emaS16):], emaS7[len(emaS7)-len(emaS16):], \
        emaS8[len(emaS8)-len(emaS16):], emaS9[len(emaS9)-len(emaS16):], emaS10[len(emaS10)-len(emaS16):], \
        emaS11[len(emaS11)-len(emaS16):], emaS12[len(emaS12)-len(emaS16):], emaS13[len(emaS13)-len(emaS16):], \
                   emaS14[len(emaS14)-len(emaS16):], emaS15[len(emaS15)-len(emaS16):], emaS16))


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M1, 0, 500)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    v = niss_Fast(rates_frame['close'])
    rates_frame['nissFast'] = [0]*(len(rates_frame['close']) - len(v)) + v
    v = niss_Slow(rates_frame['close'])
    rates_frame['nissSlow'] = [0]*(len(rates_frame['close']) - len(v)) + v
    
    rates_frame['nissOscRaw'] = [0]*69 + list(map(lambda x,y : ((x - y)/y)*100, rates_frame['nissFast'][69:], rates_frame['nissSlow'][69:]))

    rates_frame['nissOsc'] = rates_frame['nissOscRaw'].rolling(window=1).mean()
    
    v = ema(rates_frame['nissOscRaw'], 24)
    rates_frame['nissSignal'] = [0]*(len(rates_frame['close']) - len(v)) + v
    rates_frame['sma'] = rates_frame['close'].rolling(window=200).mean()
    

    return rates_frame

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        print("sell")
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 2000
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)


        
        
def run(symbol):
    lot = 0.02
    check = 0
    buy_check = 0
    buy_up = 0
    sell_up = 0

    buy = 1
    sell = 0

    while True:
        df = get_values(symbol)
        if df.iloc[-2].nissOsc < df.iloc[-2].nissSignal and check == 0:
            print(f"{df.iloc[-2].close}--{df.iloc[-2].name}")
            buy_price = df.iloc[-2].close
            result_sell = Action(symbol, lot, sell)
            check = 1
        def close(buy_price, result_sell):
            sell_price = df.iloc[-2].close
            pp = price_action(symbol, 0.2, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            if pp < -6.0:
                profit.append(-6.0)
                check = 0
                result_sell = Action_close(result_sell.order, symbol, sell, lot)

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| result_comment-->{result_sell.comment}")
                    
            if pp > 6.0:
                check = 0
                result_sell = Action_close(result_sell.order, symbol, sell, lot)

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| result_comment-->{result_sell.comment}")
                    
# GBPJPY
for symbol in ['EURUSD']: 
    run(symbol)


In [ ]:
def Action(symbol, lot, signal):
    try:
        print("sell")
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        print(price)
        deviation = 2000
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

Action(symbol, 0.02, 0)


In [ ]:

global df


def price_action(symbol, lot, ask, bid, order_type):
    buy_profit = mt5.order_calc_profit(order_type, symbol, lot, ask, bid)
    return buy_profit

def ema(s, n):
    ema = []
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)

    return ema

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df


def niss_Fast(src):
    emaF1 = ema(src, 3)
    emaF2 = ema(src, 5)
    emaF3 = ema(src, 7)
    emaF4 = ema(src, 9)
    emaF5 = ema(src, 11)
    emaF6 = ema(src, 13)
    emaF7 = ema(src, 15)
    emaF8 = ema(src, 17)
    emaF9 = ema(src, 19)
    emaF10 = ema(src, 21)
    emaF11 = ema(src, 23)
    return list(map(lambda x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11:(x1+x2+x3+x4+x5+x6+x7+x8+x9+x10+x11)/11, \
             emaF1[len(emaF1)-len(emaF11):], \
             emaF2[len(emaF2)-len(emaF11):],  emaF3[len(emaF3)-len(emaF11):], emaF4[len(emaF4)-len(emaF11):], \
            emaF5[len(emaF5)-len(emaF11):], emaF6[len(emaF6)-len(emaF11):], emaF7[len(emaF7)-len(emaF11):], \
            emaF8[len(emaF8)-len(emaF11):], emaF9[len(emaF9)-len(emaF11):], emaF10[len(emaF10)-len(emaF11):], \
            emaF11))
    
def niss_Slow(src):
    emaS1 = ema(src, 25)
    emaS2 = ema(src, 28)
    emaS3 = ema(src, 31)
    emaS4 = ema(src, 34)
    emaS5 = ema(src, 37)
    emaS6 = ema(src, 40)
    emaS7 = ema(src, 43)
    emaS8 = ema(src, 46)
    emaS9 = ema(src, 49)
    emaS10 = ema(src, 52)
    emaS11 = ema(src, 55)
    emaS12 = ema(src, 58)
    emaS13 = ema(src, 61)
    emaS14 = ema(src, 64)
    emaS15 = ema(src, 67)
    emaS16 = ema(src, 70)
    return list(map(lambda x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, x13, x14, x15, x16: \
                    (x1+x2+x3+x4+x5+x6+x7+x8+x9+x10+x11+x12+x13+x14+x15+x16)/16, \
         emaS1[len(emaS1)-len(emaS16):], \
         emaS2[len(emaS2)-len(emaS16):],  emaS3[len(emaS3)-len(emaS16):], emaS4[len(emaS4)-len(emaS16):], \
        emaS5[len(emaS5)-len(emaS16):], emaS6[len(emaS6)-len(emaS16):], emaS7[len(emaS7)-len(emaS16):], \
        emaS8[len(emaS8)-len(emaS16):], emaS9[len(emaS9)-len(emaS16):], emaS10[len(emaS10)-len(emaS16):], \
        emaS11[len(emaS11)-len(emaS16):], emaS12[len(emaS12)-len(emaS16):], emaS13[len(emaS13)-len(emaS16):], \
                   emaS14[len(emaS14)-len(emaS16):], emaS15[len(emaS15)-len(emaS16):], emaS16))


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M1, 0, 500)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    v = niss_Fast(rates_frame['close'])
    rates_frame['nissFast'] = [0]*(len(rates_frame['close']) - len(v)) + v
    v = niss_Slow(rates_frame['close'])
    rates_frame['nissSlow'] = [0]*(len(rates_frame['close']) - len(v)) + v
    
    rates_frame['nissOscRaw'] = [0]*69 + list(map(lambda x,y : ((x - y)/y)*100, rates_frame['nissFast'][69:], rates_frame['nissSlow'][69:]))

    rates_frame['nissOsc'] = rates_frame['nissOscRaw'].rolling(window=1).mean()
    
    v = ema(rates_frame['nissOscRaw'], 24)
    rates_frame['nissSignal'] = [0]*(len(rates_frame['close']) - len(v)) + v
    rates_frame['sma'] = rates_frame['close'].rolling(window=200).mean()
    

    return rates_frame

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        print("sell")
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 2000
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

symbol = "EURUSD"
def getval(symbol):
    global df
    while True:        
        df = get_values(symbol)
        time.sleep(1)
t2 = threading.Thread(target=getval, args=(symbol,)).start()
time.sleep(2)
def run(symbol):
    global lot
    lot = 0.2
    check = 0
    buy_check = 0
    buy_up = 0
    sell_up = 0
    global sell
    global buy
    global profit
    profit = []
    buy = 1
    sell = 0
    old_time = 0

    while True:
        if df.iloc[-2].nissOsc < df.iloc[-2].nissSignal and df.iloc[-2].name != old_time:
            old_time = df.iloc[-2].name
            print(f"{df.iloc[-2].close}--{df.iloc[-2].name}")
            buy_price = df.iloc[-2].close
            result_sell = Action(symbol, lot, sell)
            t1 = threading.Thread(target=close, args=(buy_price, result_sell)).start()
#             break
        time.sleep(10)
            
            
def close(buy_price, result_sell):
    while True:
        sell_price = df.iloc[-1].close
        pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
#         time.sleep(10)
        if pp <= -6.0:
            check = 0
            result_sell = Action_close(result_sell.order, symbol, sell, lot)

            if result_sell.comment == "Requote":
                result_sell = Action_close(result_sell.order, symbol, sell, lot)
                print(f"Close Symbol-->{symbol} ||| result_sell.order-->{result_sell.order} ||| result_comment-->{result_sell.comment} ||| Requoted")
            else:
                print(f"Close  Symbol-->{symbol} ||| result_sell.order-->{result_sell.order} ||| result_comment-->{result_sell.comment}")
            break

        if pp > 6.0:
            check = 0
            result_sell = Action_close(result_sell.order, symbol, sell, lot)

            if result_sell.comment == "Requote":
                result_sell = Action_close(result_sell.order, symbol, sell, lot)
                print(f"Close  Symbol-->{symbol} ||| result_sell.order-->{result_sell.order} ||| result_comment-->{result_sell.comment} ||| Requoted")
            else:
                print(f"Close  Symbol-->{symbol} ||| result_sell.order-->{result_sell.order} ||| result_comment-->{result_sell.comment}")
            break

# GBPJPY
for symbol in ['EURUSD']: 
    run(symbol)
